In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-08 21:40:48.710108: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-07-08 21:40:48.793409: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-08 21:40:50.475831: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (1.26.16) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153]
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 5,
  "iterations_threshold": 2,
  "chunk_size": 9 * 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )

In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()

In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-08 21:40:54,372 [DEBUG] [Rain] Rain is initialized
2023-07-08 21:40:54,400 [DEBUG] [Provisioner] Creating coordinator
2023-07-08 21:40:54,421 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-08 21:40:54,433 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-08 21:40:54,445 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-08 21:40:54,467 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-08 21:40:54,492 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-08 21:40:54,518 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='semi_sync')

2023-07-08 21:40:54,554 [INFO] [Provisioner] provisioner is serving
2023-07-08 21:40:54,557 [DEBUG] [Provisioner] Starting coordinator
2023-07-08 21:40:54,563 [INFO] [Coordinator] coordinator is serving
2023-07-08 21:40:54,566 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-08 21:40:54,575 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-08 21:40:54,582 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-08 21:40:54,594 [DEBUG] [Coordinator] coordinator is sending the number of workers to provisioner
2023-07-08 21:40:54,602 [DEBUG] [Provisioner] Provision requested the coordinator to get the number of workers
2023-07-08 21:40:54,604 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-08 21:40:54,620 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker_50151/
2023-07-08 21:40:54,638 [INFO] [Worker_50151] Worker is running 

Epoch 1/2
Epoch 1/2
157/157 [==============================] - 5s 10ms/step - loss: 0.7213 - accuracy: 0.7732
Epoch 2/2
Epoch 2/2
157/157 [==============================] - 1s 9ms/step - loss: 0.3100 - accuracy: 0.9046


2023-07-08 21:41:06,510 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker1
2023-07-08 21:41:06,513 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider


2023-07-08 21:41:06,537 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker3
2023-07-08 21:41:06,539 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3


...

DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-08 21:45:21,503 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-08 21:45:21,638 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
DEBUG:DeepLearning:Asynchronous update is done by worker 2
2023-07-08 21:45:21,834 [DEBUG] [DeepLearning] Iteration 4/5 complete for worker 2.
DEBUG:DeepLearning:Iteration 4/5 complete for worker 2.
2023-07-08 21:45:21,847 [DEBUG] [DeepLearning] Starting iteration 5/5
DEBUG:DeepLearning:Starting iteration 5/5
2023-07-08 21:45:21,855 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
DEBUG:DividerAmbassador:127.0.0.1:50152
2023-07-08 21:45:21,865 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration 5 to worker 2
2023-07-08 21:45:21,871 [DEBUG] [DeepL

Epoch 1/2
157/157 [==============================] - 2s 11ms/step - loss: 0.1250 - accuracy: 0.9614
Epoch 2/2
157/157 [==============================] - 2s 12ms/step - loss: 0.1048 - accuracy: 0.9683


2023-07-08 21:46:07,193 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker2
2023-07-08 21:46:07,200 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


sending data to divider


2023-07-08 21:46:07,354 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-08 21:46:07,405 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
DEBUG:DeepLearning:Asynchronous update is done by worker 2
2023-07-08 21:46:07,488 [DEBUG] [DeepLearning] Iteration 5/5 complete for worker 2.
DEBUG:DeepLearning:Iteration 5/5 complete for worker 2.
2023-07-08 21:46:07,497 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-08 21:46:07,503 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving


In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 5ms/step - loss: 0.0814 - accuracy: 0.9764

Test accuracy: 97.6%


In [10]:
del rain

2023-07-08 21:46:08,569 [INFO] [Worker_50151] Worker stopped serving on port: 50151
INFO:Worker_50151:Worker stopped serving on port: 50151
2023-07-08 21:46:08,581 [INFO] [Worker_50152] Worker stopped serving on port: 50152
INFO:Worker_50152:Worker stopped serving on port: 50152
2023-07-08 21:46:08,588 [INFO] [Worker_50153] Worker stopped serving on port: 50153
INFO:Worker_50153:Worker stopped serving on port: 50153
2023-07-08 21:46:08,596 [DEBUG] [LocalProvisioner] Workers are deleted
DEBUG:LocalProvisioner:Workers are deleted
2023-07-08 21:46:08,602 [INFO] [Provisioner] provisioner stopped serving
INFO:Provisioner:provisioner stopped serving


In [11]:
# model = create_model()
# rain = Rain(config, model)

In [12]:
# model = rain.train(X_train, y_train, strategy='sync')

In [13]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [14]:
# del rain